# CatBoost hyperparameter tuning

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Determine number of CPU cores for parallel processing (leave one core free)
n_cores = max(1, os.cpu_count() - 1)

# Load data
TRAIN_DATA_PATH = Path("data/train_data.parquet")
train_df = pd.read_parquet(TRAIN_DATA_PATH)

train_df.index = pd.to_numeric(train_df.index)

# Drop datetime features
cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop)

In [3]:
# Separate features and targets
train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

In [6]:
# Create subsets of the data for different training sizes
# chronological order is preserved, so we take the last N rows for each subset
train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

# Identify categorical columns
cat_cols = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

## Classification

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [ ]:
# Add all imports needed for classification hyperparameter tuning
import optuna
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)
from optuna.samplers import TPESampler

# Check the dates of 1k subset to ensure all data is from same month
# due to reverse order of validation split due to poor results in normal order
print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

# Define path for saving tuning visualizations
CATBOOST_TUNING_DIR_CAT = Path("hyperparameter_tuning/CatBoost/Classification")

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with extreme bad results on last 200 rows so switched the order
X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)


# Define the objective function for Optuna to optimize
def objective_catboost(trial):
    # Possible parameters to tune with ranges
    params = {
        "iterations": trial.suggest_int(
            "iterations", 100, 1000
        ),  # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "depth": trial.suggest_int(
            "depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-3, 10.0, log=True
        ),  # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float(
            "random_strength", 1e-3, 10.0, log=True
        ),  # Amount of randomness in scoring splits
        "rsm": trial.suggest_float(
            "rsm", 0.2, 1.0
        ),  # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int(
            "border_count", 64, 254
        ),  # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "objective": "Logloss",  # Binary classification objective
        "random_seed": 42,  # Fixed seed for reproducibility
        "thread_count": 1,  # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["Plain", "Ordered"]
        ),  # Boosting type
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
        ),  # Bootstrap method for sampling data
        "cat_features": cat_cols,  # Categorical features
        "verbose": False,  # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered":  # Ordered boosting
        params["grow_policy"] = "SymmetricTree"  # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )  # Growth of trees

    if params["grow_policy"] == "Lossguide":  # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int(
            "max_leaves", 3, 31
        )  # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 15, 100
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2"  # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise":  # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 15, 100
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian":  # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature", 0, 10
        )  # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.2, 1.0
        )  # Subsample ratio

    # TimeSeriesSplit to respect order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = CatBoostClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_catboost = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_catboost.optimize(
    objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_catboost)
fig1.show()

fig2 = plot_param_importances(study_catboost)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_catboost,
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_1k_history.html")
fig2.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_1k_importance.html")
fig3.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_1k_parallel.html")

[I 2026-04-25 21:30:24,117] A new study created in memory with name: no-name-bc3d7201-1115-451a-9d1e-a0533284ca7f
[I 2026-04-25 21:30:31,386] Trial 3 finished with value: 0.5872045961701134 and parameters: {'iterations': 282, 'learning_rate': 0.012606038456206, 'depth': 2, 'l2_leaf_reg': 0.007450851774954155, 'random_strength': 0.31003771048613, 'rsm': 0.239721895051001, 'border_count': 122, 'scale_pos_weight': 9.328684651199282, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'Cosine', 'subsample': 0.9518518058078105}. Best is trial 3 with value: 0.5872045961701134.
[I 2026-04-25 21:30:32,606] Trial 7 finished with value: 0.5778680514887412 and parameters: {'iterations': 250, 'learning_rate': 0.14655508418094432, 'depth': 2, 'l2_leaf_reg': 0.006800399529384482, 'random_strength': 2.0603373772466513, 'rsm': 0.6572999462201601, 'border_count': 145, 'scale_pos_weight': 9.72311324810007, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'L2', 's


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:38:59,151] Trial 121 finished with value: 0.6603043029767167 and parameters: {'iterations': 896, 'learning_rate': 0.01552275543428587, 'depth': 3, 'l2_leaf_reg': 4.826150451390922, 'random_strength': 0.11304441093642684, 'rsm': 0.4907784909559726, 'border_count': 94, 'scale_pos_weight': 8.110101378821602, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5063730186321385}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:01,719] Trial 113 finished with value: 0.6416574134677583 and parameters: {'iterations': 559, 'learning_rate': 0.027328862679486542, 'depth': 6, 'l2_leaf_reg': 3.0253425941715677, 'random_strength': 0.06264507865744205, 'rsm': 0.8395402656541269, 'border_count': 163, 'scale_pos_weight': 7.054806381693874, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.3353513708301587}. Best is trial 18 with value: 0.6740170461722186.
[I 2026-04-25 21:39:04,764] Trial 119 finished with value: 0.6658972732248595 and parameters: {'iterations': 945, 'learning_rate': 0.015903985908878888, 'depth': 6, 'l2_leaf_reg': 4.631664089474147, 'random_strength': 0.10631758001597019, 'rsm': 0.3701916759283972, 'border_count': 95, 'scale_pos_weight': 7.354205051728528, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5092149886096795}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:07,376] Trial 118 finished with value: 0.651271353167905 and parameters: {'iterations': 948, 'learning_rate': 0.015803638197727895, 'depth': 6, 'l2_leaf_reg': 2.978012446681382, 'random_strength': 0.09978175834451052, 'rsm': 0.4884420742471069, 'border_count': 68, 'scale_pos_weight': 8.072283756200141, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5086476219499025}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:09,246] Trial 116 finished with value: 0.6260456784594717 and parameters: {'iterations': 778, 'learning_rate': 0.07800990899278011, 'depth': 6, 'l2_leaf_reg': 2.751914118733835, 'random_strength': 0.11113885130321743, 'rsm': 0.8315331427131063, 'border_count': 186, 'scale_pos_weight': 7.30274620308137, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 0.16333681083508989}. Best is trial 18 with value: 0.6740170461722186.
[I 2026-04-25 21:39:17,903] Trial 115 finished with value: 0.6544160150194633 and parameters: {'iterations': 910, 'learning_rate': 0.07592376155822776, 'depth': 6, 'l2_leaf_reg': 4.018360179592434, 'random_strength': 0.09420691161288067, 'rsm': 0.6776372765479879, 'border_count': 139, 'scale_pos_weight': 8.083620227054833, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 2.891440770233683}. Best is trial 18 with value: 0.6740170461722186.
[


BEST AUC: 0.6740
BEST PARAMETERS:
best_params = {
    "iterations": 433,
    "learning_rate": 0.03554345124418867,
    "depth": 4,
    "l2_leaf_reg": 9.840089950178873,
    "random_strength": 0.02805738171539979,
    "rsm": 0.9135819869398158,
    "border_count": 64,
    "scale_pos_weight": 6.980459859158197,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 9.907283731079449,
}

--- PARAMETER IMPORTANCE ---
  l2_leaf_reg         : 0.3015
  bootstrap_type      : 0.2216
  learning_rate       : 0.1397
  border_count        : 0.1314
  rsm                 : 0.0720
  scale_pos_weight    : 0.0567
  depth               : 0.0421
  iterations          : 0.0244
  random_strength     : 0.0090
  boosting_type       : 0.0017


In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_catboost.best_params.copy()

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["thread_count"] = n_cores
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "=" * 40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("=" * 40)

BEST PARAMS: {'iterations': 433, 'learning_rate': 0.03554345124418867, 'depth': 4, 'l2_leaf_reg': 9.840089950178873, 'random_strength': 0.02805738171539979, 'rsm': 0.9135819869398158, 'border_count': 64, 'scale_pos_weight': 6.980459859158197, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 9.907283731079449, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.6740
Holdout Test AUC:     0.6839


### 10k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning
X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)


# Define the objective function for Optuna to optimize
def objective_catboost(trial):
    # Possible parameters to tune with ranges
    params = {
        "iterations": trial.suggest_int(
            "iterations", 100, 1000
        ),  # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "depth": trial.suggest_int(
            "depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-3, 10.0, log=True
        ),  # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float(
            "random_strength", 1e-3, 10.0, log=True
        ),  # Amount of randomness in scoring splits
        "rsm": trial.suggest_float(
            "rsm", 0.2, 1.0
        ),  # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int(
            "border_count", 64, 254
        ),  # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "objective": "Logloss",  # Binary classification objective
        "random_seed": 42,  # Fixed seed for reproducibility
        "thread_count": 1,  # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False,  # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["Plain", "Ordered"]
        ),  # Boosting type
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
        ),  # Bootstrap method for sampling data
        "cat_features": cat_cols,  # Categorical features
        "verbose": False,  # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered":  # Ordered boosting
        params["grow_policy"] = "SymmetricTree"  # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )  # Growth of trees

    if params["grow_policy"] == "Lossguide":  # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int(
            "max_leaves", 3, 31
        )  # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 15, 100
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2"  # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise":  # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 15, 100
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian":  # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature", 0, 10
        )  # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.2, 1.0
        )  # Subsample ratio

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = CatBoostClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()

# Create Optuna study and optimize
study_catboost = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_catboost.optimize(
    objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_catboost)
fig1.show()

fig2 = plot_param_importances(study_catboost)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_catboost,
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_10k_history.html")
fig2.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_10k_importance.html")
fig3.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_10k_parallel.html")

[I 2026-04-29 13:30:36,684] A new study created in memory with name: no-name-cd6511cc-8baa-45ce-a09a-96d83107c35b
[I 2026-04-29 13:30:48,832] Trial 6 finished with value: 0.6751274634499956 and parameters: {'iterations': 314, 'learning_rate': 0.011506198091604546, 'depth': 5, 'l2_leaf_reg': 0.003404911525637695, 'random_strength': 0.9342977029657686, 'rsm': 0.888562827308643, 'border_count': 103, 'scale_pos_weight': 3.1510306138419555, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'SymmetricTree', 'score_function': 'L2', 'subsample': 0.3157042189537843}. Best is trial 6 with value: 0.6751274634499956.
[I 2026-04-29 13:30:53,755] Trial 0 finished with value: 0.6728159957289156 and parameters: {'iterations': 353, 'learning_rate': 0.0053896007394694904, 'depth': 2, 'l2_leaf_reg': 0.3131051628354691, 'random_strength': 3.0969234266131638, 'rsm': 0.6623541590686852, 'border_count': 211, 'scale_pos_weight': 3.7025850015696236, 'boosting_type': 'Ordered', 'bootstrap_type':


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-29 13:46:44,612] Trial 200 finished with value: 0.6860996752970188 and parameters: {'iterations': 588, 'learning_rate': 0.012246079674910475, 'depth': 4, 'l2_leaf_reg': 3.3310648815972224, 'random_strength': 0.15072437218097995, 'rsm': 0.6369316919881264, 'border_count': 180, 'scale_pos_weight': 5.365110972834289, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 12, 'min_data_in_leaf': 52, 'bagging_temperature': 7.1604271193124145}. Best is trial 98 with value: 0.6875164055031054.
[I 2026-04-29 13:46:44,761] Trial 202 finished with value: 0.684647203459072 and parameters: {'iterations': 559, 'learning_rate': 0.01210241697327003, 'depth': 4, 'l2_leaf_reg': 1.1628134538904444, 'random_strength': 0.20552339318477408, 'rsm': 0.6342168818330702, 'border_count': 80, 'scale_pos_weight': 5.385692557036192, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 12, 'min_data_in_leaf': 94, 'bag


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-29 13:46:47,692] Trial 203 finished with value: 0.685852580609307 and parameters: {'iterations': 444, 'learning_rate': 0.012729602720011404, 'depth': 4, 'l2_leaf_reg': 3.377097413323856, 'random_strength': 0.26329582370604143, 'rsm': 0.732700864210718, 'border_count': 166, 'scale_pos_weight': 5.872849357190298, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 15, 'min_data_in_leaf': 41, 'bagging_temperature': 9.479532487514637}. Best is trial 98 with value: 0.6875164055031054.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-29 13:46:52,607] Trial 204 finished with value: 0.684671805917899 and parameters: {'iterations': 441, 'learning_rate': 0.01203524047031396, 'depth': 4, 'l2_leaf_reg': 2.6105087131589144, 'random_strength': 0.303517870330497, 'rsm': 0.6359480534208142, 'border_count': 175, 'scale_pos_weight': 5.7965413749197054, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 15, 'min_data_in_leaf': 52, 'bagging_temperature': 9.956143594217584}. Best is trial 98 with value: 0.6875164055031054.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-29 13:47:03,182] Trial 205 finished with value: 0.6848042935429689 and parameters: {'iterations': 750, 'learning_rate': 0.016821890011864984, 'depth': 4, 'l2_leaf_reg': 3.352558522445287, 'random_strength': 0.2762621580039346, 'rsm': 0.6425236161284021, 'border_count': 176, 'scale_pos_weight': 5.782556497473204, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 12, 'min_data_in_leaf': 48, 'bagging_temperature': 9.629308183433904}. Best is trial 98 with value: 0.6875164055031054.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6875
BEST PARAMETERS:
best_params = {
    "iterations": 502,
    "learning_rate": 0.014976757257733092,
    "depth": 8,
    "l2_leaf_reg": 0.008738906090086759,
    "random_strength": 0.6403151618824401,
    "rsm": 0.7659083035388369,
    "border_count": 162,
    "scale_pos_weight": 5.959581256258957,
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "grow_policy": "Lossguide",
    "max_leaves": 17,
    "min_data_in_leaf": 91,
    "bagging_temperature": 8.58954612632319,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.3180
  iterations          : 0.2328
  bootstrap_type      : 0.1852
  border_count        : 0.1083
  rsm                 : 0.0647
  depth               : 0.0392
  scale_pos_weight    : 0.0192
  l2_leaf_reg         : 0.0124
  boosting_type       : 0.0109
  random_strength     : 0.0092


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params.copy()

# Apply scaling trick
original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(
    f"[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["thread_count"] = n_cores
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "=" * 40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("=" * 40)

[Scaling Trick Applied] CATBOOST: Trees 502 -> 1004, LR 0.0150 -> 0.0075
BEST PARAMS: {'iterations': 1004, 'learning_rate': 0.007488378628866546, 'depth': 8, 'l2_leaf_reg': 0.008738906090086759, 'random_strength': 0.6403151618824401, 'rsm': 0.7659083035388369, 'border_count': 162, 'scale_pos_weight': 5.959581256258957, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 17, 'min_data_in_leaf': 91, 'bagging_temperature': 8.58954612632319, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.6875
Holdout Test AUC:     0.6809


### 100k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning
X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)


# Define the objective function for Optuna to optimize
def objective_catboost(trial):
    # Possible parameters to tune with ranges
    params = {
        "iterations": trial.suggest_int(
            "iterations", 100, 1000
        ),  # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "depth": trial.suggest_int(
            "depth", 2, 10
        ),  # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-8, 100.0, log=True
        ),  # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float(
            "random_strength", 1e-3, 10.0, log=True
        ),  # Amount of randomness in scoring splits
        "rsm": trial.suggest_float(
            "rsm", 0.2, 1.0
        ),  # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int(
            "border_count", 64, 254
        ),  # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "objective": "Logloss",  # Binary classification objective
        "random_seed": 42,  # Fixed seed for reproducibility
        "thread_count": 1,  # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False,  # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["Plain", "Ordered"]
        ),  # Boosting type
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
        ),  # Bootstrap method for sampling data
        "cat_features": cat_cols,  # Categorical features
        "verbose": False,  # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered":  # Ordered boosting
        params["grow_policy"] = "SymmetricTree"  # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )  # Growth of trees

    if params["grow_policy"] == "Lossguide":  # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int(
            "max_leaves", 5, 512
        )  # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 10, 500
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2"  # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise":  # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 10, 500
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian":  # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature", 0, 10
        )  # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.2, 1.0
        )  # Subsample ratio

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = CatBoostClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_catboost = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_catboost.optimize(
    objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_catboost)
fig1.show()

fig2 = plot_param_importances(study_catboost)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_catboost,
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_100k_history.html")
fig2.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_100k_importance.html")
fig3.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_100k_parallel.html")

[I 2026-04-25 22:36:16,542] A new study created in memory with name: no-name-8f7f73bb-21a7-4ba5-9617-f55581f59106
[I 2026-04-25 22:36:38,535] Trial 1 finished with value: 0.687080201567958 and parameters: {'iterations': 100, 'learning_rate': 0.011786159335307696, 'depth': 4, 'l2_leaf_reg': 0.0007079509186669808, 'random_strength': 6.215132838552753, 'rsm': 0.271697264637637, 'border_count': 213, 'scale_pos_weight': 8.421851667114398, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 19, 'score_function': 'Cosine', 'subsample': 0.2700140967854213}. Best is trial 1 with value: 0.687080201567958.
[I 2026-04-25 22:39:02,310] Trial 5 finished with value: 0.704707052164137 and parameters: {'iterations': 521, 'learning_rate': 0.028952147373694056, 'depth': 5, 'l2_leaf_reg': 2.2903231897548626e-07, 'random_strength': 0.07857635328013197, 'rsm': 0.7743655860120331, 'border_count': 139, 'scale_pos_weight': 9.421046698627677, 'boosting_type':


BEST AUC: 0.7114
BEST PARAMETERS:
best_params = {
    "iterations": 894,
    "learning_rate": 0.025302116751832124,
    "depth": 8,
    "l2_leaf_reg": 98.54551873139987,
    "random_strength": 0.030339567093011274,
    "rsm": 0.8587177152995968,
    "border_count": 185,
    "scale_pos_weight": 6.177887840704495,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bernoulli",
    "score_function": "L2",
    "subsample": 0.7842935885515135,
}

--- PARAMETER IMPORTANCE ---
  scale_pos_weight    : 0.1942
  boosting_type       : 0.1825
  bootstrap_type      : 0.1804
  depth               : 0.1623
  learning_rate       : 0.1535
  iterations          : 0.0479
  rsm                 : 0.0346
  l2_leaf_reg         : 0.0215
  random_strength     : 0.0162
  border_count        : 0.0069


In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_catboost.best_params.copy()

# Apply scaling trick
original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["thread_count"] = n_cores
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 894 -> 8940, LR 0.0253 -> 0.0025
BEST PARAMS: {'iterations': 8940, 'learning_rate': 0.0025302116751832124, 'depth': 8, 'l2_leaf_reg': 98.54551873139987, 'random_strength': 0.030339567093011274, 'rsm': 0.8587177152995968, 'border_count': 185, 'scale_pos_weight': 6.177887840704495, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.7842935885515135, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.7114
Holdout Test AUC:     0.7148


### Full data set

In [ ]:
# Parameter tuning settings
# Exteded timeout due to slower speed of CatBoost
timeout_seconds = 60 * 60 * 6
no_improvement_trials = 100

# Split the full dataset to evaluate on holdout after tuning
split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]


# Define the objective function for Optuna to optimize
def objective_catboost(trial):
    # Possible parameters to tune with ranges
    params = {
        "iterations": trial.suggest_int(
            "iterations", 100, 1000
        ),  # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "depth": trial.suggest_int(
            "depth", 2, 10
        ),  # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-8, 100.0, log=True
        ),  # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float(
            "random_strength", 1e-3, 10.0, log=True
        ),  # Amount of randomness in scoring splits
        "rsm": trial.suggest_float(
            "rsm", 0.2, 1.0
        ),  # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int(
            "border_count", 64, 254
        ),  # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "objective": "Logloss",  # Binary classification objective
        "random_seed": 42,  # Fixed seed for reproducibility
        "thread_count": 1,  # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False,  # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["Plain", "Ordered"]
        ),  # Boosting type
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
        ),  # Bootstrap method for sampling data
        "cat_features": cat_cols,  # Categorical features
        "verbose": False,  # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered":  # Ordered boosting
        params["grow_policy"] = "SymmetricTree"  # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )  # Growth of trees

    if params["grow_policy"] == "Lossguide":  # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int(
            "max_leaves", 5, 512
        )  # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 10, 500
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2"  # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise":  # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int(
            "min_data_in_leaf", 10, 500
        )  # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical(
            "score_function", ["L2", "Cosine"]
        )  # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian":  # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature", 0, 10
        )  # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.2, 1.0
        )  # Subsample ratio

    # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = CatBoostClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()

# Create Optuna study and optimize
study_catboost = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_catboost.optimize(
    objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_catboost)
fig1.show()

fig2 = plot_param_importances(study_catboost)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_catboost,
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_full_history.html")
fig2.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_full_importance.html")
fig3.write_html(CATBOOST_TUNING_DIR_CAT / "optuna_catboost_full_parallel.html")

[I 2026-04-28 17:20:10,426] A new study created in memory with name: no-name-72991ba3-b01a-4678-b7f2-6463c5252d43
[I 2026-04-28 17:23:01,829] Trial 1 finished with value: 0.7233845610572825 and parameters: {'iterations': 161, 'learning_rate': 0.02899321081302095, 'depth': 2, 'l2_leaf_reg': 3.0000503885760953e-07, 'random_strength': 0.004118298952121502, 'rsm': 0.8579705839614105, 'border_count': 131, 'scale_pos_weight': 5.259404269121885, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'SymmetricTree', 'score_function': 'Cosine', 'subsample': 0.6430578220135994}. Best is trial 1 with value: 0.7233845610572825.
[I 2026-04-28 17:32:06,540] Trial 4 finished with value: 0.7241241327141531 and parameters: {'iterations': 167, 'learning_rate': 0.05948222984540632, 'depth': 9, 'l2_leaf_reg': 0.05336792058746114, 'random_strength': 0.0010342130085206989, 'rsm': 0.38789924384080454, 'border_count': 198, 'scale_pos_weight': 9.507379256008669, 'boosting_type': 'Ordered', 'b


BEST AUC: 0.7369
BEST PARAMETERS:
best_params = {
    "iterations": 862,
    "learning_rate": 0.05036874795758855,
    "depth": 7,
    "l2_leaf_reg": 28.934704951911797,
    "random_strength": 0.5144245877609427,
    "rsm": 0.5706570280810435,
    "border_count": 105,
    "scale_pos_weight": 7.201993563101878,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bernoulli",
    "score_function": "L2",
    "subsample": 0.5675364012486861,
}

--- PARAMETER IMPORTANCE ---
  border_count        : 0.2806
  iterations          : 0.2172
  bootstrap_type      : 0.1622
  learning_rate       : 0.1494
  depth               : 0.0870
  scale_pos_weight    : 0.0760
  rsm                 : 0.0121
  l2_leaf_reg         : 0.0098
  boosting_type       : 0.0039
  random_strength     : 0.0016


In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_catboost.best_params.copy()

# Apply scaling trick
original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["thread_count"] = n_cores
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 862 -> 8620, LR 0.0504 -> 0.0050
BEST PARAMS: {'iterations': 8620, 'learning_rate': 0.005036874795758855, 'depth': 7, 'l2_leaf_reg': 28.934704951911797, 'random_strength': 0.5144245877609427, 'rsm': 0.5706570280810435, 'border_count': 105, 'scale_pos_weight': 7.201993563101878, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.5675364012486861, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.7369
Holdout Test AUC:     0.7274


## Regression

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [16]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_parallel.html")


[I 2026-04-26 09:35:05,085] A new study created in memory with name: no-name-428bbc8f-82dc-4583-a4f1-8efe0902bb16
[I 2026-04-26 09:35:06,788] Trial 1 finished with value: 0.3206080735305823 and parameters: {'iterations': 260, 'learning_rate': 0.05764775623977107, 'depth': 2, 'l2_leaf_reg': 1.9588073144766103, 'random_strength': 0.0034399568829632135, 'rsm': 0.2838689946143098, 'border_count': 134, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 2.890491659130583}. Best is trial 1 with value: 0.3206080735305823.
[I 2026-04-26 09:35:08,642] Trial 5 finished with value: 0.31825801262520625 and parameters: {'iterations': 216, 'learning_rate': 0.012761438892139924, 'depth': 3, 'l2_leaf_reg': 0.468539643734418, 'random_strength': 0.027204782574556764, 'rsm': 0.9722745523370238, 'border_count': 253, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 4, 'min_data_in_leaf': 87, 'bagging_tempe


BEST RMSE: 0.3144
BEST PARAMETERS:
best_params = {
    "iterations": 976,
    "learning_rate": 0.00976818825318037,
    "depth": 4,
    "l2_leaf_reg": 0.027265370477102587,
    "random_strength": 0.7031697546586838,
    "rsm": 0.8385396695909164,
    "border_count": 201,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "Cosine",
    "bagging_temperature": 9.530476491737797,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.9508
  iterations          : 0.0140
  bootstrap_type      : 0.0125
  rsm                 : 0.0085
  border_count        : 0.0082
  boosting_type       : 0.0026
  random_strength     : 0.0014
  depth               : 0.0013
  l2_leaf_reg         : 0.0008


In [17]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params

best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)

BEST PARAMS: {'iterations': 976, 'learning_rate': 0.00976818825318037, 'depth': 4, 'l2_leaf_reg': 0.027265370477102587, 'random_strength': 0.7031697546586838, 'rsm': 0.8385396695909164, 'border_count': 201, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 9.530476491737797, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.3144
Holdout Test RMSE:     0.3340


### 10k

In [18]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_parallel.html")


[I 2026-04-26 10:05:54,557] A new study created in memory with name: no-name-4c4da6da-5cca-4dc7-b6c2-205bdd9b0da0
[I 2026-04-26 10:06:10,100] Trial 2 finished with value: 0.31121998114057103 and parameters: {'iterations': 859, 'learning_rate': 0.1749200343284882, 'depth': 3, 'l2_leaf_reg': 0.0036528914748255453, 'random_strength': 0.7634974526460588, 'rsm': 0.37865276175104623, 'border_count': 152, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'SymmetricTree', 'score_function': 'L2', 'subsample': 0.3157131881566521}. Best is trial 2 with value: 0.31121998114057103.
[I 2026-04-26 10:06:17,167] Trial 7 finished with value: 0.2979299940087425 and parameters: {'iterations': 611, 'learning_rate': 0.02065640778824637, 'depth': 2, 'l2_leaf_reg': 2.0431013480789515, 'random_strength': 1.0124176312513955, 'rsm': 0.7573138246806337, 'border_count': 211, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 22, 'min_data_in_leaf': 35


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:34:50,733] Trial 140 finished with value: 0.29700052068038396 and parameters: {'iterations': 908, 'learning_rate': 0.012763956553074614, 'depth': 4, 'l2_leaf_reg': 0.3088398529333473, 'random_strength': 2.68677326425155, 'rsm': 0.744147359668745, 'border_count': 103, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 7.545412966046143}. Best is trial 40 with value: 0.2966091160106081.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:34:51,296] Trial 138 finished with value: 0.29710364822773117 and parameters: {'iterations': 908, 'learning_rate': 0.012419620505449983, 'depth': 5, 'l2_leaf_reg': 0.28556984446035416, 'random_strength': 4.932583605491949, 'rsm': 0.5544106827664926, 'border_count': 107, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.363044521531694}. Best is trial 40 with value: 0.2966091160106081.
[I 2026-04-26 10:34:58,815] Trial 141 finished with value: 0.29704943717356647 and parameters: {'iterations': 977, 'learning_rate': 0.012533188328160407, 'depth': 4, 'l2_leaf_reg': 0.2823093271389073, 'random_strength': 4.663953365628512, 'rsm': 0.7488369395614154, 'border_count': 105, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.340680518623314}. Best is trial 40 with value: 0.2966091160106081.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:35:09,286] Trial 142 finished with value: 0.29691641849952805 and parameters: {'iterations': 907, 'learning_rate': 0.014650464000978436, 'depth': 4, 'l2_leaf_reg': 0.2713179363210563, 'random_strength': 3.8327513231803443, 'rsm': 0.744867101720851, 'border_count': 153, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 6.112443024703257}. Best is trial 40 with value: 0.2966091160106081.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:35:23,165] Trial 139 finished with value: 0.2968049791601305 and parameters: {'iterations': 969, 'learning_rate': 0.01460958035398821, 'depth': 5, 'l2_leaf_reg': 0.19832002184547623, 'random_strength': 0.3567191173008058, 'rsm': 0.738050846104574, 'border_count': 103, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.356827165173819}. Best is trial 40 with value: 0.2966091160106081.
[I 2026-04-26 10:37:15,984] Trial 13 finished with value: 0.7359240728067847 and parameters: {'iterations': 4897, 'learning_rate': 0.016370061296226476, 'depth': 6, 'l2_leaf_reg': 0.2727482933935999, 'random_strength': 3.5964693066900386, 'rsm': 0.3821606351295279, 'border_count': 167, 'scale_pos_weight': 4.733863711713026, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 9.125809324314888}. Best is trial 13 with value: 0.7359240728067847.
[I 2026-04-26 10:37:45,594] Trial 13


BEST RMSE: 0.2966
BEST PARAMETERS:
best_params = {
    "iterations": 544,
    "learning_rate": 0.027650691306981594,
    "depth": 5,
    "l2_leaf_reg": 0.5148645656737443,
    "random_strength": 0.1893828349911231,
    "rsm": 0.7999433598790255,
    "border_count": 194,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 8.170909952414544,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.8100
  bootstrap_type      : 0.0861
  rsm                 : 0.0309
  iterations          : 0.0254
  l2_leaf_reg         : 0.0204
  border_count        : 0.0087
  boosting_type       : 0.0085
  depth               : 0.0063
  random_strength     : 0.0037


In [19]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 544 -> 1088, LR 0.0277 -> 0.0138
BEST PARAMS: {'iterations': 1088, 'learning_rate': 0.013825345653490797, 'depth': 5, 'l2_leaf_reg': 0.5148645656737443, 'random_strength': 0.1893828349911231, 'rsm': 0.7999433598790255, 'border_count': 194, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 8.170909952414544, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.2966
Holdout Test RMSE:     0.3122


### 100k

In [20]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning


X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_parallel.html")


[I 2026-04-26 10:38:01,962] A new study created in memory with name: no-name-92b41a08-583e-4d57-a1ba-70a8b359d5af
[I 2026-04-26 10:39:09,495] Trial 2 finished with value: 0.3024753664951711 and parameters: {'iterations': 324, 'learning_rate': 0.006994633234659066, 'depth': 5, 'l2_leaf_reg': 7.690569983928778e-08, 'random_strength': 0.20204959872204886, 'rsm': 0.35533999411198103, 'border_count': 64, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.2594852262145646}. Best is trial 2 with value: 0.3024753664951711.
[I 2026-04-26 10:39:59,681] Trial 7 finished with value: 0.300819159021708 and parameters: {'iterations': 324, 'learning_rate': 0.18525489468588302, 'depth': 2, 'l2_leaf_reg': 1.0389424660506077, 'random_strength': 0.0011900467641136516, 'rsm': 0.8297319520832989, 'border_count': 92, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 5.994185547847424}. Best is trial 7 with v


BEST RMSE: 0.2993
BEST PARAMETERS:
best_params = {
    "iterations": 993,
    "learning_rate": 0.010876805260096451,
    "depth": 9,
    "l2_leaf_reg": 25.36922452973143,
    "random_strength": 1.1801448166458588,
    "rsm": 0.7742077921478168,
    "border_count": 217,
    "boosting_type": "Plain",
    "bootstrap_type": "MVS",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 299,
    "score_function": "L2",
    "subsample": 0.269182776220681,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.6736
  depth               : 0.1173
  bootstrap_type      : 0.0564
  border_count        : 0.0458
  boosting_type       : 0.0346
  l2_leaf_reg         : 0.0251
  iterations          : 0.0227
  rsm                 : 0.0131
  random_strength     : 0.0113


In [21]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick

original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 993 -> 9930, LR 0.0109 -> 0.0011
BEST PARAMS: {'iterations': 9930, 'learning_rate': 0.001087680526009645, 'depth': 9, 'l2_leaf_reg': 25.36922452973143, 'random_strength': 1.1801448166458588, 'rsm': 0.7742077921478168, 'border_count': 217, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 299, 'score_function': 'L2', 'subsample': 0.269182776220681, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.2993
Holdout Test RMSE:     0.2984


### Full data set

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*6
no_improvement_trials = 100

# Split the full dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_parallel.html")


[I 2026-04-27 23:24:28,151] A new study created in memory with name: no-name-a3b63e56-87a5-48e2-803e-dc4f5535af76
[I 2026-04-27 23:32:13,048] Trial 3 finished with value: 0.23628511160752488 and parameters: {'iterations': 619, 'learning_rate': 0.04844712559068459, 'depth': 2, 'l2_leaf_reg': 0.0026055026833229447, 'random_strength': 0.13792467349428414, 'rsm': 0.49159237704289344, 'border_count': 233, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 179, 'min_data_in_leaf': 301, 'bagging_temperature': 4.78375028998772}. Best is trial 3 with value: 0.23628511160752488.
[I 2026-04-27 23:32:43,534] Trial 6 finished with value: 0.23615785523719815 and parameters: {'iterations': 479, 'learning_rate': 0.08500023054406174, 'depth': 3, 'l2_leaf_reg': 2.3108243104242096e-05, 'random_strength': 3.1676270686965147, 'rsm': 0.43889952384348696, 'border_count': 78, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max


BEST RMSE: 0.2339
BEST PARAMETERS:
best_params = {
    "iterations": 878,
    "learning_rate": 0.019516220841303295,
    "depth": 9,
    "l2_leaf_reg": 2.9869534553308876e-07,
    "random_strength": 0.028533049262069288,
    "rsm": 0.41630399524229533,
    "border_count": 167,
    "boosting_type": "Plain",
    "bootstrap_type": "Bernoulli",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 482,
    "score_function": "L2",
    "subsample": 0.8907108810335598,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.3742
  depth               : 0.2352
  random_strength     : 0.1914
  iterations          : 0.0716
  bootstrap_type      : 0.0453
  rsm                 : 0.0413
  border_count        : 0.0377
  l2_leaf_reg         : 0.0024
  boosting_type       : 0.0009


In [7]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 878 -> 8780, LR 0.0195 -> 0.0020
BEST PARAMS: {'iterations': 8780, 'learning_rate': 0.0019516220841303295, 'depth': 9, 'l2_leaf_reg': 2.9869534553308876e-07, 'random_strength': 0.028533049262069288, 'rsm': 0.41630399524229533, 'border_count': 167, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 482, 'score_function': 'L2', 'subsample': 0.8907108810335598, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.2339
Holdout Test RMSE:     0.2832
